# Reproduction of One Contribution
**Which Contribution is Being Reproduced**: I am writing the code for the "Inverse Calibration" method from Section 3 of the paper. This means I will change group probabilities into inverse-scaled targets, calculate the error margins using the Taylor expansion, and find the best model using an optimization solver.

**Evaluation Metric**: The paper judges success using `Relative Accuracy`. This is calculated by taking the accuracy of our Inverse Calibration model and dividing it by the accuracy of a normal SVM model that knows all the individual labels.

In [1]:
import numpy as np
import cvxpy as cp
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import os

# Set a random seed
np.random.seed(42)

# Load the data we saved earlier
X_train = np.load('partB/data/X_train.npy')
y_train = np.load('partB/data/y_train.npy')
X_test = np.load('partB/data/X_test.npy')
y_test = np.load('partB/data/y_test.npy')

### What this code block computes
This block loads the preprocessed training and test data we saved in Task 2.1 and sets the random seed. It ensures we use exactly the same split and scaling for fair comparison with the supervised SVM baseline.

### Forming the Probabilistic Groups
Before running the main algorithm, we need to group the data. The paper groups the data into sets of a fixed size $k$. I am using $k=16$, which was one of the sizes tested in the paper. We will calculate the percentage of $y=1$ labels in each group. This percentage acts as our group probability $p_k$.
Reference: Section 3, where $p_k = |\{i \in S_k : y_i = 1\}| / |S_k|$ is defined.

In [2]:
k = 16
n_samples = X_train.shape[0]
n_groups = int(np.ceil(n_samples / k))
indices = np.arange(n_samples)
np.random.shuffle(indices)

# Group the data indices into chunks of size k
groups = [indices[i*k:min((i+1)*k, n_samples)] for i in range(n_groups)]

p_group = []
X_mean = []

# Calculate the probability and the average feature values for each group
for g in groups:
    prob = np.sum(y_train[g] == 1) / len(g)
    p_group.append(prob)
    X_mean.append(np.mean(X_train[g], axis=0))

p_group = np.array(p_group)
X_mean = np.array(X_mean)

### Applying Inverse Calibration and SVR Optimization
Now we do the main math from the paper. We change the group probabilities $p_{group}$ into new target numbers $y_{target}$ using the inverse logit formula $y = -\log(1/p - 1)$. We also calculate the allowed error margin $\epsilon_i$ for each group.
After that, we use the CVXPY library to solve the math problem exactly as described in the paper's "Primal Problem".
Reference: Section 3: "Primal Problem" and "Taylor expansion of order 1".

In [3]:
# Keep probabilities slightly away from 0 and 1 so the math doesn't break
eps_clip = 1 / n_samples
p_clipped = np.clip(p_group, eps_clip, 1 - eps_clip)

# Inverse scaling step (from Section 3)
y_targets = -np.log(1 / p_clipped - 1)

# Calculate the dynamic error margin using the Taylor expansion
base_epsilon = 0.1
eps_dynamic = base_epsilon / (p_clipped * (1 - p_clipped))

# Setup the Support Vector Regression using CVXPY
m = len(groups)
d = X_train.shape[1]
C = 1.0

w = cp.Variable(d)
b = cp.Variable()
xi = cp.Variable(m, nonneg=True)
xi_star = cp.Variable(m, nonneg=True)

# We want to minimize the complexity of the model and the errors
objective = cp.Minimize(0.5 * cp.sum_squares(w) + C * cp.sum(xi + xi_star))

# The model's prediction for a group should be close to the target value
constraints = [
    X_mean @ w + b >= y_targets - eps_dynamic - xi,
    X_mean @ w + b <= y_targets + eps_dynamic + xi_star
]

prob = cp.Problem(objective, constraints)
prob.solve()

# Test our Inverse model
w_opt, b_opt = w.value, b.value
y_pred_inverse = np.sign(X_test @ w_opt + b_opt)
acc_inverse = accuracy_score(y_test, y_pred_inverse)

# Train a normal, fully-supervised SVM for comparison
svm = SVC(kernel='linear')
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)
acc_svm = accuracy_score(y_test, y_pred_svm)

relative_accuracy = acc_inverse / acc_svm

os.makedirs('partB/results', exist_ok=True)
np.save('partB/results/inverse_calibration_results.npy',
        np.array([acc_inverse, acc_svm, relative_accuracy]))

print(f"Inverse Calibration Accuracy: {acc_inverse:.4f}")
print(f"Normal SVM Accuracy: {acc_svm:.4f}")
print(f"Relative Accuracy: {relative_accuracy:.4f}")

Inverse Calibration Accuracy: 0.8947
Normal SVM Accuracy: 0.9766
Relative Accuracy: 0.9162


### What this code block computes
This is the core of Inverse Calibration. It clips probabilities, computes the inverse targets y_targets using the sigmoid inverse from the paper, calculates the group-specific error margins ε_i via the Taylor expansion, solves the exact primal SVR problem with CVXPY, predicts on the test set, compares against a fully-supervised linear SVM, computes the paper’s relative accuracy metric, and saves the results.